### Import Libraries

In [2]:
# Import required libraries for data preprocessing
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import joblib
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")

Libraries imported successfully!


### Load and Explore Dataset

In [3]:
# Load stock prices dataset
df = pd.read_csv('2) Stock Prices Data Set.csv')

print("Original Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Original Dataset Shape: (497472, 7)

First 5 rows:


,symbol,date,open,high,low,close,volume
0,AAL,2014-01-02,25.0700,25.8200,25.0600,25.3600,8998943
1,AAPL,2014-01-02,79.3828,79.5756,78.8601,79.0185,58791957
2,AAP,2014-01-02,110.3600,111.8800,109.2900,109.7400,542711
3,ABBV,2014-01-02,52.1200,52.3300,51.5200,51.9800,4569061
4,ABC,2014-01-02,70.1100,70.2300,69.4800,69.8900,1148391


In [4]:
print("Dataset Info:")
print(df.info())

print("\nMissing Values Count:")
print(df.isnull().sum())

print("\nMissing Values Percentage:")
print((df.isnull().sum() / len(df)) * 100)

print("\nColumn Names:")
print(df.columns.tolist())

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 497472 entries, 0 to 497471
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   symbol  497472 non-null  str    
 1   date    497472 non-null  str    
 2   open    497461 non-null  float64
 3   high    497464 non-null  float64
 4   low     497464 non-null  float64
 5   close   497472 non-null  float64
 6   volume  497472 non-null  int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 26.6 MB
None

Missing Values Count:
symbol     0
date       0
open      11
high       8
low        8
close      0
volume     0
dtype: int64

Missing Values Percentage:
symbol    0.000000
date      0.000000
open      0.002211
high      0.001608
low       0.001608
close     0.000000
volume    0.000000
dtype: float64

Column Names:
['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']


### Handle Missing Data

In [5]:
# Handle missing values in numerical columns (open, high, low)
numerical_cols = ['open', 'high', 'low']

# Method 1: Fill with mean
df_mean = df.copy()
for col in numerical_cols:
    if df_mean[col].isnull().any():
        mean_val = df_mean[col].mean()
        df_mean[col].fillna(mean_val, inplace=True)
        print(f"Filled {col} with mean: {mean_val:.2f}")

# Method 2: Fill with median
df_median = df.copy()
for col in numerical_cols:
    if df_median[col].isnull().any():
        median_val = df_median[col].median()
        df_median[col].fillna(median_val, inplace=True)
        print(f"Filled {col} with median: {median_val:.2f}")

# Method 3: Drop rows with missing values
df_dropped = df.dropna(subset=numerical_cols)
print(f"\nRows after dropping missing values: {df_dropped.shape[0]} (original: {df.shape[0]})")

Filled open with mean: 86.35
Filled high with mean: 87.13
Filled low with mean: 85.55
Filled open with median: 64.97
Filled high with median: 65.56
Filled low with median: 64.35

Rows after dropping missing values: 497461 (original: 497472)


### Encode Categorical Variables

In [6]:
# Use mean-filled dataset for encoding
df_encoded = df_mean.copy()

# 1. Process 'date' column: convert to datetime and extract numerical features
df_encoded['date'] = pd.to_datetime(df_encoded['date'])
df_encoded['day_of_week'] = df_encoded['date'].dt.dayofweek
df_encoded['month'] = df_encoded['date'].dt.month
df_encoded['day'] = df_encoded['date'].dt.day
df_encoded = df_encoded.drop('date', axis=1)
print("Added date features: day_of_week, month, day")

# 2. Label Encode 'symbol' column
le = LabelEncoder()
df_encoded['symbol'] = le.fit_transform(df_encoded['symbol'].astype(str))
print(f"\nLabel Encoded 'symbol' (first 5 classes): {list(le.classes_[:5])}...")

# 3. One-Hot Encode 'symbol' (alternative method)
df_onehot = df_encoded.copy()
df_onehot = pd.get_dummies(df_onehot, columns=['symbol'], drop_first=True)
print(f"\nOne-Hot Encoded shape: {df_onehot.shape} (original: {df_encoded.shape})")

Added date features: day_of_week, month, day

Label Encoded 'symbol' (first 5 classes): ['A', 'AAL', 'AAP', 'AAPL', 'ABBV']...

One-Hot Encoded shape: (497472, 512) (original: (497472, 9))


### Normalize/Standardize Numerical Features

In [7]:
# Select numerical columns to scale
numerical_cols_to_scale = ['open', 'high', 'low', 'close', 'volume', 'day_of_week', 'month', 'day']

# Method 1: Normalization (Min-Max Scaling to [0,1])
minmax_scaler = MinMaxScaler()
df_normalized = df_encoded.copy()
df_normalized[numerical_cols_to_scale] = minmax_scaler.fit_transform(df_encoded[numerical_cols_to_scale])
print("=== Normalization (Min-Max) Applied ===")
print(f"Min values (should be 0): {df_normalized[numerical_cols_to_scale].min().round(2).values}")
print(f"Max values (should be 1): {df_normalized[numerical_cols_to_scale].max().round(2).values}")

# Method 2: Standardization (Z-Score, mean=0, std=1)
standard_scaler = StandardScaler()
df_standardized = df_encoded.copy()
df_standardized[numerical_cols_to_scale] = standard_scaler.fit_transform(df_encoded[numerical_cols_to_scale])
print("\n=== Standardization (Z-Score) Applied ===")
print(f"Mean (should be ~0): {df_standardized[numerical_cols_to_scale].mean().round(2).values}")
print(f"Std (should be ~1): {df_standardized[numerical_cols_to_scale].std().round(2).values}")

=== Normalization (Min-Max) Applied ===
Min values (should be 0): [0. 0. 0. 0. 0. 0. 0. 0.]
Max values (should be 1): [1. 1. 1. 1. 1. 1. 1. 1.]

=== Standardization (Z-Score) Applied ===
Mean (should be ~0): [ 0. -0. -0. -0. -0. -0. -0.  0.]
Std (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1.]


### Create Target Variable

In [8]:
# Create target for supervised learning: price_movement (1=price up next day, 0=down)
df_final = df_standardized.copy()
df_final['price_movement'] = (df_final['close'].shift(-1) > df_final['close']).astype(int)
df_final = df_final.dropna(subset=['price_movement'])

print("Target variable 'price_movement' created")
print(f"Target distribution: {df_final['price_movement'].value_counts().to_dict()}")
print(f"Final dataset shape: {df_final.shape}")

Target variable 'price_movement' created
Target distribution: {0: 249985, 1: 247487}
Final dataset shape: (497472, 10)


### Split Dataset into Training and Testing Sets

In [9]:
# Split into features (X) and target (y)
X = df_final.drop('price_movement', axis=1)
y = df_final['price_movement']

# 80-20 train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== Dataset Split Complete ===")
print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Testing set: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"Training target distribution: {y_train.value_counts().to_dict()}")
print(f"Testing target distribution: {y_test.value_counts().to_dict()}")

=== Dataset Split Complete ===
Training set: 397977 samples, 9 features
Testing set: 99495 samples, 9 features
Training target distribution: {0: 199988, 1: 197989}
Testing target distribution: {0: 49997, 1: 49498}


### Save Preprocessed Data

In [ ]:
# Save processed data and scalers
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
joblib.dump(minmax_scaler, 'minmax_scaler.pkl')
joblib.dump(standard_scaler, 'standard_scaler.pkl')
joblib.dump(le, 'label_encoders.pkl')
print("Preprocessed data saved successfully!")

Preprocessed data saved successfully!
